# Evaluation

I trained the model for 10 epochs on the text8 corpus and compare it's outputs to the Gensim implementation of Word2vec trained with the same hyperparameters to validate it's capabilities. Both models seem to perform comaparably well and can identify similar words and some analogies. However, a larger corpus is likely needed to get better embeddings.

In [65]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

In [80]:
# Model wrapper for easier access to embeddings
class Model:
    def __init__(self, embedding, vocabulary):
        self.embedding = embedding
        self.norm_embedding = embedding / np.linalg.norm(embedding, axis=1, keepdims=True)
        self.index_to_word = vocabulary
        self.word_to_index = { word: idx for idx, word in enumerate(vocabulary) }

    def __getitem__(self, word):
        return self.embedding[self.word_to_index[word]]

    def similar_words(self, word, top_k=5):        
        similarities = np.dot(self.norm_embedding, self.norm_embedding[self.word_to_index[word]])
        top_indices = np.argsort(similarities)[::-1][1:top_k+1]
        return [self.index_to_word[i] for i in top_indices]
    
    def closest_words(self, embedding, top_k=5):
        similarities = np.dot(self.norm_embedding, embedding / np.linalg.norm(embedding))
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(self.index_to_word[i], similarities[i]) for i in top_indices]
    
    def analogy(self, a, b, c, top_k=5):
        target = self[b] - self[a] + self[c]
        candidates = self.closest_words(target, top_k=top_k + 10)
        excluded = {a, b, c}

        results = []
        for word, score in candidates:
            if word not in excluded:
                results.append((word, score))
            if len(results) == top_k:
                break
        return results
    
    def similarity(self, word1, word2):
        return np.dot(self.norm_embedding[self.word_to_index[word1]], self.norm_embedding[self.word_to_index[word2]])

In [88]:
# Load our trained model
data = np.load("model/epoch_10.npz")
model = Model(data['embedding'], data['vocabulary'])

In [68]:
gensim_w2v = Word2Vec(
    sentences=LineSentence("data/text8.txt"),
    vector_size=100,
    window=5,
    sample=1e-5,
    min_count=1,
    sg=1,
    negative=5,
    epochs=10,
    alpha=0.025,
    min_alpha=0.025
)

## Similar words
The model resulted in similar embeddings numbers, months, units, countries, etc. Unrelated words are mixed in occasionally.

In [79]:
words = ["october", "monday", "winter", # time
         "two", "sixth", "number", # numerals
         "kilogram", "ampere", "watt", # units
         "java", "python", # programming languages
         "oxygen", "carbon", "uranium", # elements
         "spain", "washington", "nile", # geography
         "car", "mercedes", "ferrari", # cars
         "king", "queen", "prince", # royals
         "sheep", "rhinoceros", "parrot" # animals
         ]

rows = []
for word in words:
    closest_words = model.similar_words(word, 5)
    gensim_closest_words = [ w for w, _ in gensim_w2v.wv.most_similar(word, topn=5) ]

    rows.append({
        "Word": word,
        "Top 5 closest words": ", ".join(closest_words),
        "Top 5 closest words (Gensim)": ", ".join(gensim_closest_words),
    })
    
    rows[-1] if gensim_closest_words else "(OOV)"
with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(rows)
        .style
        .hide(axis="index")
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
        .set_properties(subset=["Top 5 closest words"], **{"text-align": "left"})
        .set_properties(subset=["Top 5 closest words (Gensim)"], **{"text-align": "left"})
    )


Word,Top 5 closest words,Top 5 closest words (Gensim)
october,"november, june, december, july, april","august, february, june, january, march"
monday,"friday, day, sunday, wednesday, rosenmotag","filers, sauntering, friday, sunday, jailyn"
winter,"spoonbridge, summer, ungenial, martisor, qingming","snowsqualls, slushly, spring, snowfalls, zensen"
two,"four, zero, five, seven, one","five, six, zero, nine, four"
sixth,"fourth, eighth, fifth, seventh, twelfth","fifth, ninth, seventh, fourth, tenth"
number,"sizes, fraction, representing, smaller, corresponding","inconceivably, least, numbers, padovan, significiant"
kilogram,"kilograms, kilogramme, billionths, megajoules, grams","kilograms, kilogramme, kgm, kilopond, lbm"
ampere,"coulomb, amperes, si, radian, dbu","coulomb, amperes, volt, ammeter, coulombs"
watt,"lumens, kilowatt, dbk, drawbar, jinjer","lumens, boulton, kilowatt, dbw, dbk"
java,"javadoc, scripting, integra, mdk, gfortran","doxygen, ddoc, javadoc, gcj, applets"


## Analogies
On a small hand-crafted set of 26 analogies, the custom model solves several relations (for example, **man:king::woman:queen**, comparative forms, and some plurals), but performance is inconsistent across categories such as capitals and some verb transformations.

In [78]:
analogies = [
    # family / gender
    ("man", "king", "woman", "queen"),
    ("boy", "prince", "girl", "princess"),
    ("husband", "wife", "man", "woman"),

    # countries / capitals
    ("france", "paris", "germany", "berlin"),
    ("italy", "rome", "france", "paris"),
    ("spain", "madrid", "italy", "rome"),
    ("japan", "tokyo", "china", "beijing"),
    ("england", "london", "france", "paris"),

    # comparatives / superlatives
    ("small", "smaller", "large", "larger"),
    ("fast", "faster", "slow", "slower"),
    ("young", "younger", "old", "older"),
    ("high", "higher", "low", "lower"),
    ("long", "longer", "short", "shorter"),

    # morphological: verb tense
    ("walk", "walked", "talk", "talked"),
    ("play", "played", "work", "worked"),
    ("look", "looking", "read", "reading"),

    # morphological: adverb formation
    ("quick", "quickly", "slow", "slowly"),
    ("easy", "easily", "happy", "happily"),

    # morphological: plurals
    ("car", "cars", "dog", "dogs"),
    ("book", "books", "house", "houses"),
    ("cat", "cats", "bird", "birds"),
    ("tree", "trees", "flower", "flowers"),
    ("river", "rivers", "mountain", "mountains"),
    ("city", "cities", "country", "countries"),


    # morphological: opposites with prefixes
    ("happy", "unhappy", "fair", "unfair"),
    ("known", "unknown", "clear", "unclear"),
]

analogy_rows = []
for a, b, c, expected in analogies:
    predictions = model.analogy(a, b, c, top_k=5)
    predicted_words = [w for w, _ in predictions]

    analogy_rows.append({
        "Analogy": f"{a} : {b} :: {c} : ?",
        "Expected": expected,
        "Top 5 closest words": ", ".join(predicted_words)
    })

with pd.option_context("display.max_colwidth", None):
    df_analogies = pd.DataFrame(analogy_rows)

    # Add Success@5 for your model (if not already present)
    if "Success@5" not in df_analogies.columns:
        df_analogies["Success@5"] = df_analogies.apply(
            lambda r: "✅" if r["Expected"] in [w.strip() for w in r["Top 5 closest words"].split(",")] else "❌",
            axis=1
        )

    # Add Gensim analogy predictions and success
    gensim_top5 = []
    gensim_success = []
    for a, b, c, expected in analogies:
        try:
            g_pred = [w for w, _ in gensim_w2v.wv.most_similar(positive=[b, c], negative=[a], topn=5)]
        except KeyError:
            g_pred = []

        gensim_top5.append(", ".join(g_pred) if g_pred else "(OOV)")
        gensim_success.append("✅" if expected in g_pred else "❌")

    df_analogies["Top 5 closest words (Gensim)"] = gensim_top5
    df_analogies["Success@5 (Gensim)"] = gensim_success

    display(
        df_analogies[
            ["Analogy", "Expected", "Success@5", "Top 5 closest words", "Success@5 (Gensim)", "Top 5 closest words (Gensim)"]
        ]
        .style
        .hide(axis="index")
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
        .set_properties(
            subset=["Analogy", "Top 5 closest words", "Top 5 closest words (Gensim)"],
            **{"text-align": "left"}
        )
    )

Analogy,Expected,Success@5,Top 5 closest words,Success@5 (Gensim),Top 5 closest words (Gensim)
man : king :: woman : ?,queen,✅,"queen, metapontus, techichpotzin, throne, legitimist",❌,"scottsboro, ermengarde, eschiva, betrothed, hersilia"
boy : prince :: girl : ?,princess,✅,"queen, crown, princess, overlordship, thored",✅,"princess, heiress, dagmar, stadtholder, callbeck"
husband : wife :: man : ?,woman,❌,"groundskeeper, reodor, delarge, grimhild, exotically",❌,"harlot, brazen, hawwah, jrrt, rizpah"
france : paris :: germany : ?,berlin,❌,"merzbau, munich, hietzing, colarossi, burgtheater",✅,"berlin, altenberg, munich, nste, cottbus"
italy : rome :: france : ?,paris,❌,"moreever, witzin, childebert, mainila, rueil",❌,"bourbons, decembrist, napoleon, varennes, foral"
spain : madrid :: italy : ?,rome,❌,"ricordi, munich, tosca, museo, burgtheater",❌,"bergamo, bologna, cagliari, turin, munich"
japan : tokyo :: china : ?,beijing,❌,"taipei, seinan, kunitachi, mctba, veszprem",❌,"koreas, taipei, chongqing, chiang, guangdong"
england : london :: france : ?,paris,✅,"moscow, paris, brussels, kongreso, rhin",❌,"chirac, raffarin, mitterrand, vladivostok, gaulle"
small : smaller :: large : ?,larger,❌,"stomatogastric, number, antennas, accessible, desecrations",✅,"larger, bigger, allots, fewer, than"
fast : faster :: slow : ?,slower,✅,"slower, plasticize, apochromats, artchitectures, vitascope",✅,"slower, outpace, improvements, downforce, drivability"


## WordSim353
The WordSim353 evaluation measures how well cosine similarities from the learned embeddings align with human similarity judgments, using Spearman correlation. Both my implementation and Gensim show a moderate correlation with human scores (around **0.6**), indicating comparable performance on this benchmark.


In [85]:
ws = pd.read_csv("data/wordsim_353.csv")
w1 = ws["Word 1"].str.lower()
w2 = ws["Word 2"].str.lower()
human_corr = ws["Human (mean)"]


pred = np.array([model.similarity(a, b) for a, b in zip(w1, w2)], dtype=float)
gensim_pred = np.array([gensim_w2v.wv.similarity(a, b) for a, b in zip(w1, w2)], dtype=float)

def correlation(pred):
    return float(pd.Series(human_corr).corr(pd.Series(pred), method="spearman"))

results = pd.DataFrame(
    {
        "Model": ["my word2vec", "gensim"],
        "Spearman correlation": [correlation(pred), correlation(gensim_pred)],
    }
)

display(
    results.style
    .hide(axis="index")
    .format({"Spearman correlation": "{:.4f}"})
)


Model,Spearman correlation
my word2vec,0.5846
gensim,0.6258


## Analogies dataset
I used the dataset used for the evaluation in the paper [https://www.fit.vut.cz/person/imikolov/public/rnnlm/word-test.v1.txt](https://www.fit.vut.cz/person/imikolov/public/rnnlm/word-test.v1.txt). The accuracy is quite low for models, indicating the limiations of the dataset.


In [87]:
file_analogies = []
with open("data/analogies.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip().lower()
        if not line or line.startswith(":"):
            continue
        parts = line.split()
        if len(parts) >= 4:
            a, b, c, expected = parts[:4]
            file_analogies.append((a, b, c, expected))

# Evaluate Accuracy@5 for both models
my_success = 0
my_total = 0
gensim_success = 0
gensim_total = 0

for a, b, c, expected in file_analogies:
    # my model
    try:
        pred_my = [w for w, _ in model.analogy(a, b, c, top_k=5)]
        my_success += int(expected in pred_my)
        my_total += 1
    except KeyError:
        pass  # OOV for my model

    # gensim model
    try:
        pred_g = [w for w, _ in gensim_w2v.wv.most_similar(positive=[b, c], negative=[a], topn=5)]
        gensim_success += int(expected in pred_g)
        gensim_total += 1
    except KeyError:
        pass  # OOV for gensim

accuracy_summary = pd.DataFrame({
    "Model": ["my word2vec", "gensim"],
    "Accuracy@5": [
        my_success / my_total if my_total else np.nan,
        gensim_success / gensim_total if gensim_total else np.nan,
    ],
})

display(
    accuracy_summary.style
    .hide(axis="index")
    .format({"Accuracy@5": "{:.2%}"})
)

Model,Accuracy@5
my word2vec,18.73%
gensim,26.98%
